# 🏺 Hieroglyph NLP Pipeline — Kaggle v8
### Transliteration → Dictionary → spaCy → Arabic Translation → Sentiment
---
> Running on **Kaggle** — GPU P100 (16GB)

## ▶ Cell 0 — Install Dependencies

In [1]:
import subprocess, sys

def pip(*pkgs):
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs),
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f'⚠️  {result.stderr[-300:]}')

pip('protobuf==3.20.3')
pip('transformers==4.44.2', 'sentencepiece', 'torch', 'requests', 'spacy', 'scipy', 'numpy', 'accelerate', 'safetensors==0.4.3')
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

print('✅ All dependencies installed')
print('👉 Session → Restart & Run All')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 79.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All dependencies installed
👉 Session → Restart & Run All


## 📁 Cell 1 — Dataset Paths

In [2]:
# ── Paths ثابتة من Kaggle Input ─────────────────────────────────────
GARDINER_PATH  = '/kaggle/input/datasets/mo3azkhaled/gardiner-sign/Update_gardiner_sign.csv'
DICT_PATH      = '/kaggle/input/datasets/mo3azkhaled/egyptian-dictionary/egyptian_dictionary.csv'
INTENTION_PATH = '/kaggle/input/datasets/mo3azkhaled/intention-dataset/intention_dataset.csv'

import os
for label, path in [('Gardiner', GARDINER_PATH),
                    ('Dictionary', DICT_PATH),
                    ('Intention', INTENTION_PATH)]:
    exists = '✅' if os.path.exists(path) else '❌ NOT FOUND'
    print(f'  {exists}  {label}: {path}')


  ✅  Gardiner: /kaggle/input/datasets/mo3azkhaled/gardiner-sign/Update_gardiner_sign.csv
  ✅  Dictionary: /kaggle/input/datasets/mo3azkhaled/egyptian-dictionary/egyptian_dictionary.csv
  ✅  Intention: /kaggle/input/datasets/mo3azkhaled/intention-dataset/intention_dataset.csv


## 📖 Cell 2 — Gardiner Sign List

In [3]:
import pandas as pd

df = pd.read_csv(GARDINER_PATH)
GARDINER_MAP = {}
for _, row in df.iterrows():
    code = str(row['code']).strip().lower()
    if not code or code == 'nan':
        continue
    GARDINER_MAP[code] = {
        'phonetic': str(row['phonetic']).strip() if pd.notna(row['phonetic']) else '',
        'meaning' : str(row['meaning']).strip()  if pd.notna(row['meaning'])  else '',
        'unicode' : str(row['unicode']).strip()   if pd.notna(row['unicode'])  else '',
    }

print(f'✅ Gardiner Sign List loaded: {len(GARDINER_MAP)} signs')
print(f'   Signs with phonetic: {sum(1 for v in GARDINER_MAP.values() if v["phonetic"])}')


✅ Gardiner Sign List loaded: 819 signs
   Signs with phonetic: 224


In [4]:
print('Sample signs:')
for code in ['g17', 'n35', 'd21', 'o1', 'n5', 'f35', 'r4']:
    if code in GARDINER_MAP:
        v = GARDINER_MAP[code]
        print(f'  {code.upper():5} -> phonetic: {v["phonetic"]:8} | {v["meaning"]:30} {v["unicode"]}')
    else:
        print(f'  {code.upper():5} -> NOT FOUND')


Sample signs:
  G17   -> phonetic: m        | Owl                            𓅓
  N35   -> phonetic: n        | Water ripple                   𓈖
  D21   -> phonetic: r        | Mouth                          𓂋
  O1    -> phonetic: pr       | House plan                     𓉐
  N5    -> phonetic: ra       | Sun disk                       𓇳
  F35   -> phonetic: nfr      | Heart and windpipe             𓄤
  R4    -> phonetic: Htp      | Bread loaf on mat              𓊵


## 📚 Cell 3 — Egyptian Dictionary

In [5]:
import pandas as pd
from collections import defaultdict

df_dict = pd.read_csv(DICT_PATH)
_raw = defaultdict(list)
for _, row in df_dict.iterrows():
    key = str(row['transliteration']).strip()
    val = str(row['english']).strip()
    if not key or key == 'nan' or not val or val == 'nan':
        continue
    if val not in _raw[key]:
        _raw[key].append(val)

EGYPTIAN_DICT = dict(_raw)
print(f'✅ Egyptian Dictionary loaded: {len(EGYPTIAN_DICT)} entries')


✅ Egyptian Dictionary loaded: 1352 entries


In [6]:
for w in ['nfr', 'ra', 'Htp', 'rn', 'nTr', 'pr', 'zA']:
    meanings = EGYPTIAN_DICT.get(w, ['NOT FOUND'])
    print(f'  {w:10} -> {" | ".join(meanings[:3])}')


  nfr        -> be good / beautiful / perfect | good / beautiful / perfect | perfect / good
  ra         -> sun god Ra / Re
  Htp        -> offering / peace / satisfaction | be satisfied / rest | offer / be at peace
  rn         -> name / soul name
  nTr        -> god
  pr         -> house / estate
  zA         -> NOT FOUND


## ⚙️ Cell 4 — Transliteration Engine + spaCy

In [7]:
import re
import spacy

nlp_spacy = spacy.load('en_core_web_sm')
print('✅ spaCy loaded')


def extract_core_meaning(meanings) -> str:
    text  = meanings[0] if isinstance(meanings, list) else str(meanings)
    first = re.split(r'[/|]', text)[0].strip()
    return re.sub(r'^be ', '', first).strip().lower()


def build_sentence_spacy(core_meanings: list) -> str:
    if not core_meanings: return ''
    if len(core_meanings) == 1: return core_meanings[0]
    doc   = nlp_spacy(' '.join(core_meanings))
    nouns = [t.text for t in doc if t.pos_ in ('NOUN', 'PROPN')]
    verbs = [t.text for t in doc if t.pos_ == 'VERB']
    adjs  = [t.text for t in doc if t.pos_ == 'ADJ']
    poss  = {'name','son','daughter','house','heart','brother','sister','father','mother','lord'}
    if verbs and nouns: return f'{verbs[0]} the {nouns[0]}'
    if len(nouns) >= 2:
        return f'my {nouns[0]} is {nouns[1]}' if nouns[0].lower() in poss else f'{nouns[0]} of {nouns[1]}'
    if nouns and adjs: return f'the {adjs[0]} {nouns[0]}'
    if adjs: return f'it is {adjs[0]}'
    return ' '.join(core_meanings)


def transliterate(gardiner_codes: list) -> dict:
    codes = [c.lower().strip() for c in gardiner_codes]
    phonetics, glyphs, sign_meanings, unknown = [], [], [], []
    for c in codes:
        if c in GARDINER_MAP:
            info = GARDINER_MAP[c]
            phonetics.append(info['phonetic']); glyphs.append(info['unicode']); sign_meanings.append(info['meaning'])
        else:
            phonetics.append('?'); glyphs.append('□'); sign_meanings.append('unknown'); unknown.append(c)

    ph_clean = [p for p in phonetics if p and p != '?']
    full     = ''.join(ph_clean)

    token_results, core_meanings = [], []
    i = 0
    while i < len(ph_clean):
        matched = False
        for length in range(min(4, len(ph_clean)-i), 0, -1):
            combined = ''.join(ph_clean[i:i+length])
            if combined in EGYPTIAN_DICT:
                ml = EGYPTIAN_DICT[combined]
                core = extract_core_meaning(ml)
                token_results.append({'phonetic':combined,'meaning':' | '.join(ml),'core':core,'found':True})
                core_meanings.append(core); i += length; matched = True; break
        if not matched:
            token_results.append({'phonetic':ph_clean[i],'meaning':f'[{ph_clean[i]}]','core':ph_clean[i],'found':False}); i += 1

    candidates = [full] if full else []
    for size in range(len(ph_clean), 0, -1):
        for start in range(len(ph_clean)-size+1):
            w = ''.join(ph_clean[start:start+size])
            if w and w not in candidates: candidates.append(w)

    found_words = []
    for c in candidates:
        if c in EGYPTIAN_DICT:
            ml = EGYPTIAN_DICT[c]
            found_words.append({'transliteration':c,'meaning':' | '.join(ml),'confidence':'high' if c==full else 'partial'})

    high = [w for w in found_words if w['confidence']=='high']
    best = high[0] if high else (found_words[0] if found_words else None)

    return {
        'input_codes':codes,'glyphs':glyphs,'glyph_str':' '.join(glyphs),
        'per_sign':list(zip(codes,phonetics,sign_meanings)),
        'phonetics':phonetics,'phonetic_str':' '.join(ph_clean),
        'assembled':full,'token_results':token_results,'found_words':found_words,
        'unknown_codes':unknown,'best_meaning':best['meaning'] if best else None,
        'sentence':build_sentence_spacy(core_meanings),'core_meanings':core_meanings,
    }

print('✅ Transliteration engine ready')


✅ spaCy loaded
✅ Transliteration engine ready


In [8]:
for codes, label in [
    (['o1'],              'pr   -> house'),
    (['f35'],             'nfr  -> beautiful'),
    (['n5','r8'],         'ra nTr -> sun god'),
    (['d21','n35','n5'],  'rn ra  -> my name is sun'),
    (['r4','x8','a42'],   'Htp di nswt -> offering'),
]:
    r = transliterate(codes)
    print(f'  {label}')
    print(f'    Tokens  : {[(t["phonetic"],t["core"]) for t in r["token_results"]]}')
    print(f'    Sentence: {r["sentence"]}')
    print()


  pr   -> house
    Tokens  : [('pr', 'house')]
    Sentence: house

  nfr  -> beautiful
    Tokens  : [('nfr', 'good')]
    Sentence: good

  ra nTr -> sun god
    Tokens  : [('ra', 'sun god ra'), ('nTr', 'god')]
    Sentence: sun of god

  rn ra  -> my name is sun
    Tokens  : [('rn', 'name'), ('ra', 'sun god ra')]
    Sentence: my name is sun

  Htp di nswt -> offering
    Tokens  : [('Htp', 'offering'), ('di', 'give')]
    Sentence: offering give



## 🌍 Cell 5 — Arabic Translation (Seed-X-PPO-7B)

In [9]:
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast
import torch
from safetensors import safe_open
import transformers.modeling_utils as _mu

# Monkey-patch: إصلاح safetensors metadata bug على Kaggle
_orig_safe_open = safe_open
class _PatchedSafeOpen:
    def __init__(self, path, framework, device="cpu"):
        self._f = _orig_safe_open(path, framework=framework, device=device)
    def metadata(self):
        m = self._f.metadata()
        return m if (m is not None and m.get("format")) else {"format": "pt"}
    def keys(self): return self._f.keys()
    def get_tensor(self, k): return self._f.get_tensor(k)
    def __enter__(self): return self
    def __exit__(self, *a): pass

_mu.safe_open = _PatchedSafeOpen

SEED_X_MODEL = 'ByteDance-Seed/Seed-X-PPO-7B'
print(f'Loading {SEED_X_MODEL} ...')
print('\u26a0\ufe0f  Requires ~16GB VRAM \u2014 Kaggle P100 OK')

seed_x_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    SEED_X_MODEL,
    trust_remote_code = True,
)
if seed_x_tokenizer.pad_token is None:
    seed_x_tokenizer.pad_token = seed_x_tokenizer.eos_token

seed_x_model = AutoModelForCausalLM.from_pretrained(
    SEED_X_MODEL,
    torch_dtype       = torch.bfloat16,
    device_map        = "auto",
    trust_remote_code = True,
)
seed_x_model.eval()
print('\u2705 Seed-X-PPO-7B loaded')


def _prepare_inputs(prompt: str):
    inputs = seed_x_tokenizer(prompt, return_tensors="pt")
    inputs.pop("token_type_ids", None)  # الموديل مش بيقبل token_type_ids
    return {k: v.to(seed_x_model.device) for k, v in inputs.items()}


def translate_to_arabic(english_text: str) -> str:
    if not english_text or english_text.startswith('['): return ''
    try:
        prompt  = 'Translate the following English text into Arabic:\n' + english_text + ' <ar>'
        inputs  = _prepare_inputs(prompt)
        with torch.no_grad():
            outputs = seed_x_model.generate(
                **inputs,
                max_new_tokens = 128,
                do_sample      = False,
                pad_token_id   = seed_x_tokenizer.eos_token_id,
            )
        decoded = seed_x_tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens = True,
        )
        return decoded.strip()
    except Exception as e:
        return f'[error: {e}]'


def translate_batch(texts: list, target_lang: str = 'ar') -> list:
    if not texts: return []
    results = []
    for t in texts:
        try:
            prompt  = 'Translate the following English text into Arabic:\n' + t + ' <' + target_lang + '>'
            inputs  = _prepare_inputs(prompt)
            with torch.no_grad():
                outputs = seed_x_model.generate(
                    **inputs,
                    max_new_tokens = 128,
                    do_sample      = False,
                    pad_token_id   = seed_x_tokenizer.eos_token_id,
                )
            decoded = seed_x_tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens = True,
            )
            results.append(decoded.strip())
        except Exception as e:
            results.append(f'[error: {e}]')
    return results


print('\u2705 translate_to_arabic() ready')
print('\u2705 translate_batch()     ready')


Loading ByteDance-Seed/Seed-X-PPO-7B ...
⚠️  Requires ~16GB VRAM — Kaggle P100 OK


tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'LlamaTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/15.0G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Seed-X-PPO-7B loaded
✅ translate_to_arabic() ready
✅ translate_batch()     ready


In [ ]:
tests = ['my name is sun','son of Ra','offering which the king gives',
        'the good god','house of god','it is beautiful','give the offering']
print('English -> Arabic (Seed-X-PPO-7B):')
print()
for en, ar in zip(tests, translate_batch(tests)):
    print(f'  EN: {en}')
    print(f'  AR: {ar}')
    print()


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


English -> Arabic (Seed-X-PPO-7B):



2026-03-18 19:50:48.962231: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773863449.214616      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773863449.284845      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773863449.975361      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773863449.975414      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773863449.975417      55 computation_placer.cc:177] computation placer alr

  EN: my name is sun
  AR: اسمي سون.

  EN: son of Ra
  AR: ابن را.

  EN: offering which the king gives
  AR: العرض الذي يقدمه الملك.

  EN: the good god
  AR: الإله الجيد.

  EN: house of god
  AR: بيت الله.

  EN: it is beautiful
  AR: إنه جميل.

  EN: give the offering
  AR: أعطي القربان.



## 💭 Cell 6 — Sentiment + Intention

In [11]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch, numpy as np, pandas as pd

print('Loading Twitter RoBERTa...')
SENTIMENT_MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment'
sent_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_NAME)
sent_model     = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL_NAME)
sent_model.eval()
SENTIMENT_LABELS = ['negative', 'neutral', 'positive']
print('✅ Sentiment model loaded')


def analyze_sentiment(text: str) -> tuple:
    if not text or text.startswith('['): return 'neutral', 0.5
    try:
        inputs = sent_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            scores = torch.softmax(sent_model(**inputs).logits, dim=1).numpy()[0]
        idx = int(np.argmax(scores))
        return SENTIMENT_LABELS[idx], round(float(scores[idx]), 3)
    except:
        return 'neutral', 0.5


df_intent = pd.read_csv(INTENTION_PATH)
INTENTION_MAP = {}
for _, row in df_intent.iterrows():
    intent_en = str(row['intention_en']).strip()
    INTENTION_MAP[intent_en] = {
        'arabic'  : str(row['intention_ar']).strip(),
        'keywords': set(kw.strip().lower() for kw in str(row['keywords']).split(',')),
    }
print(f'✅ Intention dataset loaded: {len(INTENTION_MAP)} intentions')


def detect_intention(text: str, phonetics: str = '') -> tuple:
    words  = set((text + ' ' + phonetics).lower().split())
    scores = {i: len(words & d['keywords']) for i, d in INTENTION_MAP.items() if words & d['keywords']}
    if not scores: return 'descriptive', 'وصفي', {}
    best = max(scores, key=scores.get)
    return best, INTENTION_MAP[best]['arabic'], scores

print('✅ detect_intention() ready')


Loading Twitter RoBERTa...


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

✅ Sentiment model loaded
✅ Intention dataset loaded: 139 intentions
✅ detect_intention() ready


In [12]:
for text in ['my name is sun','offering of the king','god of the beautiful house',
             'death and darkness','beloved son of Ra','give life and peace']:
    label, score = analyze_sentiment(text)
    intent_en, intent_ar, _ = detect_intention(text)
    emoji = '😊' if label=='positive' else '😞' if label=='negative' else '😐'
    print(f'  {emoji} [{label:8}] ({score}) | {text}')
    print(f'     Intention: {intent_en} — {intent_ar}')
    print()


  😐 [neutral ] (0.609) | my name is sun
     Intention: resurrection — البعث والقيامة

  😐 [neutral ] (0.791) | offering of the king
     Intention: funerary_formula — صيغة جنائزية

  😊 [positive] (0.689) | god of the beautiful house
     Intention: hymn — تسبيح وترنيم

  😞 [negative] (0.518) | death and darkness
     Intention: sorrow — الحزن والألم

  😊 [positive] (0.728) | beloved son of Ra
     Intention: royal_titulary — ألقاب ملكية

  😊 [positive] (0.764) | give life and peace
     Intention: thanksgiving — شكر وامتنان



## 🚀 Cell 7 — Full NLP Pipeline

In [13]:
def run_nlp_pipeline(gardiner_codes: list, verbose: bool = True) -> dict:
    trans        = transliterate(gardiner_codes)
    best_meaning = trans['best_meaning']
    sentence     = trans['sentence']

    if best_meaning:   english, method = best_meaning, 'dictionary'
    elif sentence:     english, method = sentence,     'spacy-nlp'
    else:              english, method = f'[unknown: {trans["assembled"]}]', 'none'

    arabic           = translate_to_arabic(sentence or english)
    sent_text        = (sentence + ' ' + (best_meaning or '')).strip()
    sentiment, score = analyze_sentiment(sent_text)
    intent_en, intent_ar, intent_detail = detect_intention(sent_text, trans['phonetic_str'])

    result = {**trans,
        'english':english,'arabic':arabic,'trans_method':method,
        'sentiment':sentiment,'sent_score':score,
        'intention_en':intent_en,'intention_ar':intent_ar,'intent_detail':intent_detail,
    }

    if verbose:
        print('='*62)
        print('  🏺  HIEROGLYPH NLP PIPELINE')
        print('='*62)
        print(f'  Input codes  : {gardiner_codes}')
        print(f'  Glyphs       : {trans["glyph_str"]}')
        print('\n  Per sign:')
        for code, ph, meaning in trans['per_sign']:
            print(f'    {code.upper():6} -> {(ph or "(det.)"):10} | {meaning}')
        print('\n  Tokens:')
        for t in trans['token_results']:
            print(f'    [{"OK" if t["found"] else "?"}] {t["phonetic"]:8} -> {t["meaning"][:40]}')
        if trans['found_words']:
            print('\n  Word matches:')
            for w in trans['found_words'][:3]:
                print(f'    [{w["confidence"]:7}] {w["transliteration"]:12} = {w["meaning"][:40]}')
        emoji = '😊' if sentiment=='positive' else '😞' if sentiment=='negative' else '😐'
        print(f'\n  Sentence     : {sentence}')
        print(f'  English      : {english} [{method}]')
        print(f'  Arabic       : {arabic}')
        print(f'\n  Sentiment    : {emoji} {sentiment} (score={score})')
        print(f'  Intention EN : {intent_en}')
        print(f'  Intention AR : {intent_ar}')
        print('='*62)

    return result


for codes, label in [(['d21','n35','n5'],'rn ra'),
                     (['r4','x8','a42'],'Htp di nsw'),
                     (['n5','r8','f35'],'ra nTr nfr')]:
    print(f'TEST: {label}')
    run_nlp_pipeline(codes)
    print()


TEST: rn ra


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  🏺  HIEROGLYPH NLP PIPELINE
  Input codes  : ['d21', 'n35', 'n5']
  Glyphs       : 𓂋 𓈖 𓇳

  Per sign:
    D21    -> r          | Mouth
    N35    -> n          | Water ripple
    N5     -> ra         | Sun disk

  Tokens:
    [OK] rn       -> name / soul name
    [OK] ra       -> sun god Ra / Re

  Word matches:
    [partial] rn           = name / soul name
    [partial] r            = mouth | to / toward / concerning
    [partial] n            = to / for / of

  Sentence     : my name is sun
  English      : name / soul name [dictionary]
  Arabic       : اسمي سون.

  Sentiment    : 😐 neutral (score=0.801)
  Intention EN : rebirth
  Intention AR : التجدد والبعث

TEST: Htp di nsw


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  🏺  HIEROGLYPH NLP PIPELINE
  Input codes  : ['r4', 'x8', 'a42']
  Glyphs       : 𓊵 𓏕 𓀯

  Per sign:
    R4     -> Htp        | Bread loaf on mat
    X8     -> di         | Conical bread
    A42    -> (det.)     | Seated king holding flail

  Tokens:
    [OK] Htp      -> offering / peace / satisfaction | be sat
    [OK] di       -> give / given

  Word matches:
    [partial] Htp          = offering / peace / satisfaction | be sat
    [partial] di           = give / given

  Sentence     : offering give
  English      : offering / peace / satisfaction | be satisfied / rest | offer / be at peace | bread / offerings [dictionary]
  Arabic       : عرض منحة.

  Sentiment    : 😊 positive (score=0.667)
  Intention EN : offering
  Intention AR : تقديم قربان

TEST: ra nTr nfr
  🏺  HIEROGLYPH NLP PIPELINE
  Input codes  : ['n5', 'r8', 'f35']
  Glyphs       : 𓇳 𓊹 𓄤

  Per sign:
    N5     -> ra         | Sun disk
    R8     -> nTr        | Flag
    F35    -> nfr        | Heart and windpipe

  Tok

## 🧪 Cell 8 — Mock Tests

In [14]:
MOCK_EXAMPLES = {
    'funerary_text'   : {'description': 'نص جنائزي',     'codes': ['r4','x8','a42','n5','r8','f35','n35','n5']},
    'royal_cartouche' : {'description': 'كارتوش ملكي',   'codes': ['a42','g17','n35','d21','n5']},
    'offering_formula': {'description': 'صيغة قربان',    'codes': ['r4','x8','a42','f35','n35','g43']},
    'simple_word'     : {'description': 'house',          'codes': ['o1']},
    'name_of_ra'      : {'description': 'my name is sun', 'codes': ['d21','n35','n5']},
}

for name, ex in MOCK_EXAMPLES.items():
    print(f'  [{name}] {ex["description"]}')
    r = run_nlp_pipeline(ex['codes'], verbose=False)
    emoji = '😊' if r['sentiment']=='positive' else '😞' if r['sentiment']=='negative' else '😐'
    print(f'    Glyphs   : {r["glyph_str"]}')
    print(f'    Sentence : {r["sentence"]}')
    print(f'    English  : {r["english"]}')
    print(f'    Arabic   : {r["arabic"]}')
    print(f'    Sentiment: {emoji} {r["sentiment"]} ({r["sent_score"]})')
    print(f'    Intention: {r["intention_en"]} — {r["intention_ar"]}')
    print()


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  [funerary_text] نص جنائزي


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


    Glyphs   : 𓊵 𓏕 𓀯 𓇳 𓊹 𓄤 𓈖 𓇳
    Sentence : offering the sun
    English  : offering / peace / satisfaction | be satisfied / rest | offer / be at peace | bread / offerings
    Arabic   : تقديم الشمس.
    Sentiment: 😊 positive (0.66)
    Intention: offering — تقديم قربان

  [royal_cartouche] كارتوش ملكي


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


    Glyphs   : 𓀯 𓅓 𓈖 𓂋 𓇳
    Sentence : monument of sun
    English  : monument | monument / stela
    Arabic   : نصب الشمس.
    Sentiment: 😐 neutral (0.894)
    Intention: royal_building — بناء ملكي

  [offering_formula] صيغة قربان


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


    Glyphs   : 𓊵 𓏕 𓀯 𓄤 𓈖 𓅱
    Sentence : offering the time
    English  : time / now | Nun / primordial waters | the time / moment
    Arabic   : تقديم الوقت.
    Sentiment: 😐 neutral (0.903)
    Intention: creation_myth — أسطورة الخلق

  [simple_word] house


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


    Glyphs   : 𓉐
    Sentence : house
    English  : house / estate
    Arabic   : منزل.
    Sentiment: 😐 neutral (0.655)
    Intention: marriage — الزواج

  [name_of_ra] my name is sun
    Glyphs   : 𓂋 𓈖 𓇳
    Sentence : my name is sun
    English  : name / soul name
    Arabic   : اسمي سون.
    Sentiment: 😐 neutral (0.801)
    Intention: rebirth — التجدد والبعث



## 🔗 Cell 9 — Integration مع الـ CV Model

In [15]:
def full_pipeline_from_image(image_path: str,
                              cv_predict_fn=None,
                              mock_codes: list = None,
                              verbose: bool = True) -> dict:
    print(f'Processing: {image_path}\n')
    if cv_predict_fn is not None:
        cv_results     = cv_predict_fn(image_path, show=False)
        gardiner_codes = [r[1].lower() for r in cv_results]
        confidences    = [r[2] for r in cv_results]
        print(f'  CV detected {len(gardiner_codes)} glyphs')
        print(f'  Codes: {gardiner_codes}')
        print(f'  Conf : {[round(c,2) for c in confidences]}')
    elif mock_codes is not None:
        gardiner_codes = [c.lower() for c in mock_codes]
        print(f'  Using mock codes: {gardiner_codes}')
    else:
        raise ValueError('Provide cv_predict_fn or mock_codes')

    if not gardiner_codes:
        return {'error': 'No glyphs detected'}

    nlp = run_nlp_pipeline(gardiner_codes, verbose=verbose)
    return {
        'image':image_path, 'n_glyphs':len(gardiner_codes),
        'codes':gardiner_codes, 'glyphs':nlp['glyph_str'],
        'phonetics':nlp['phonetic_str'], 'sentence':nlp['sentence'],
        'english':nlp['english'], 'arabic':nlp['arabic'],
        'sentiment':nlp['sentiment'],
        'intention_en':nlp['intention_en'], 'intention_ar':nlp['intention_ar'],
    }


# Demo
result = full_pipeline_from_image(
    image_path = 'stela.jpg',
    mock_codes = ['r4','x8','a42','f35','n35','n5','r8'],
    verbose    = True,
)
print('\nFINAL SUMMARY:')
for k, v in result.items():
    print(f'  {k:12}: {v}')


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processing: stela.jpg

  Using mock codes: ['r4', 'x8', 'a42', 'f35', 'n35', 'n5', 'r8']
  🏺  HIEROGLYPH NLP PIPELINE
  Input codes  : ['r4', 'x8', 'a42', 'f35', 'n35', 'n5', 'r8']
  Glyphs       : 𓊵 𓏕 𓀯 𓄤 𓈖 𓇳 𓊹

  Per sign:
    R4     -> Htp        | Bread loaf on mat
    X8     -> di         | Conical bread
    A42    -> (det.)     | Seated king holding flail
    F35    -> nfr        | Heart and windpipe
    N35    -> n          | Water ripple
    N5     -> ra         | Sun disk
    R8     -> nTr        | Flag

  Tokens:
    [OK] Htp      -> offering / peace / satisfaction | be sat
    [OK] di       -> give / given
    [OK] nfr      -> be good / beautiful / perfect | good / b
    [OK] n        -> to / for / of
    [OK] ra       -> sun god Ra / Re
    [OK] nTr      -> god

  Word matches:
    [partial] Htp          = offering / peace / satisfaction | be sat
    [partial] di           = give / given
    [partial] nfr          = be good / beautiful / perfect | good / b

  Sentence     :